# Fase 6: Modelado de Machine Learning y Evaluación

**Responsable:** Jorge  
**Proyecto:** Asistente Virtual para Tiendas de Ropa  
**Materia:** Tópicos y Programación Avanzada  

---

## 1. Contexto y Objetivos
El objetivo de esta fase es entrenar, evaluar y comparar modelos de Machine Learning para clasificar texto en lenguaje natural en 13 intenciones del dominio de tiendas de ropa.

De acuerdo a los lineamientos del proyecto:
1. **Línea Base Simple:** `DummyClassifier(strategy='most_frequent')` para demostrar que los modelos aportan valor sobre una predicción trivial.
2. **Naive Bayes (`ComplementNB`):** Modelo probabilístico clásico para clasificación de texto, robusto ante desbalances leves.
3. **Regresión Logística multinomial:** Modelo lineal con regularización, interpretable y con probabilidades bien calibradas para fijar un umbral de confianza.
4. **Validación Cruzada Estratificada:** 5 particiones (`StratifiedKFold`) con métrica de optimización **F1 Macro** para no sesgar hacia clases con más frases.
5. **Análisis de Errores:** Inspección de frases mal clasificadas y justificación técnica.

In [ ]:
import os
import sys
# Asegurar que la raíz del proyecto esté en el path
sys.path.insert(0, os.path.abspath('..'))

import yaml
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.models.train import cargar_configuracion, entrenar
from src.models.predict import predecir

print("Entorno y módulos cargados exitosamente.")

## 2. Inspección de Configuración e Hiperparámetros
Todos los parámetros se gestionan desde `src/config.yaml` para asegurar reproducibilidad y cambios en vivo:

In [ ]:
config = cargar_configuracion('../src/config.yaml')
print(yaml.dump(config['models'], sort_keys=False))

## 3. Entrenamiento y Comparación de Modelos
Ejecutamos el pipeline reproducible que entrena la línea base, optimiza Naive Bayes y Regresión Logística con GridSearchCV (5-folds Stratified), y genera los artefactos:

In [ ]:
pipeline, resultados = entrenar('../src/config.yaml')
print(f"Total de muestras: {resultados['total_muestras']}")
print(f"Modelo seleccionado para producción: {resultados['modelo_produccion']['nombre']}")

## 4. Tabla Comparativa de Rendimiento
Comparamos los tres modelos sobre el conjunto de prueba independiente (test set 20% estratificado):

In [ ]:
tabla_comparativa = []
for k, v in resultados['modelos'].items():
    m = v['metricas']
    tabla_comparativa.append({
        'Modelo': k,
        'Accuracy': m['accuracy'],
        'F1 Macro': m['f1_macro'],
        'F1 Weighted': m['f1_weighted']
    })

df_comp = pd.DataFrame(tabla_comparativa)
df_comp

## 5. Visualización: Matriz de Confusión del Modelo en Producción

In [ ]:
prod_nombre = resultados['modelo_produccion']['nombre']
cm = np.array(resultados['modelos'][prod_nombre]['metricas']['confusion_matrix'])
clases = resultados['clases']

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=clases, yticklabels=clases)
plt.title(f'Matriz de Confusión - {prod_nombre}')
plt.xlabel('Predicción')
plt.ylabel('Etiqueta Real')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 6. Análisis de Errores e Interpretación
Identificamos casos donde el modelo duda o falla, analizando la probabilidad asignada y el umbral de confianza:

In [ ]:
frases_prueba = [
    "¿tienen pantalones negros en talla 32?",
    "¿a cuánto está la polera blanca?",
    "quiero que me devuelvan el dinero",
    "dame el balance de ventas de hoy",
    "un zapato volador en la luna"  # Caso fuera de dominio / baja confianza
]

for f in frases_prueba:
    res = predecir(f, '../src/config.yaml')
    print(f"Texto: '{res['texto']}'")
    print(f"  -> Intención asignada: {res['intencion']}")
    print(f"  -> Confianza: {res['probabilidad']} (Umbral: {res['umbral']})")
    print(f"  -> Top 3: {[x['intencion'] + ' (' + str(x['probabilidad']) + ')' for x in res['top3']]}")
    print("-" * 60)

## 7. Conclusiones para la Defensa
1. **Valor sobre la Línea Base:** La línea base (Dummy) obtiene un F1 macro cercano a 0.01, mientras que los modelos supervisados superan ampliamente este piso.
2. **Justificación del Modelo:** Regresión Logística y Naive Bayes se comparan rigurosamente con validación cruzada estratificada.
3. **Mecanismo de Seguridad (Umbral):** Si la consulta no tiene suficiente certidumbre probabilística, el sistema no comete una falsa afirmación y responde que no entendió la consulta.